In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import chi2_contingency
from scipy.stats.contingency import association
from scipy.stats import probplot


sns.set_palette("Set2")
sns.set_style("whitegrid")


In [ ]:
def univariate_numerical_analysis(numerical_frame: pd.DataFrame ,column:str, new_index:str):
    
    numerical_data = numerical_frame.loc[:,column]

    mean_val = round(numerical_data.mean(),2)
    median_val = round(numerical_data.median(),2)
    std_val = round(numerical_data.std(),2)
    range_val = round(numerical_data.max() - numerical_data.min(),2)  
    iqr_val = round(numerical_data.quantile(0.75)-numerical_data.quantile(0.25),2)
    skew_val = round(numerical_data.skew(),2)
    kurtosis_val = round(numerical_data.kurtosis(),2)
    coefficient_of_variance_val = round(((std_val / mean_val) if mean_val != 0 else 0),2) 
    mode_val = numerical_data.mode().tolist()    
    mode_val_string = ""

    for _ in mode_val:
        mode_val_string += str(format(_, ".2f")) + ","



    output_frame = pd.DataFrame(
        {"Mean": [mean_val],
         "Median": [median_val],
         "Mode": [mode_val],
         "Standard Deviation": [std_val],
         "Range": [range_val],
         "Inter-Quartile Range": [iqr_val],
         "Skewness": [skew_val],
         "Kurtosis": [kurtosis_val],
         "Coefficient of Variance": [coefficient_of_variance_val]})
    output_frame.index = [new_index]
   
    return(output_frame)

In [ ]:
def iqr_outlier_strip(numerical_frame: pd.DataFrame,column:str):
    
    q1 = np.percentile(numerical_frame[column], 25)
    q3 = np.percentile(numerical_frame[column], 75)
    iqr = q3 - q1
    lower_bound = q1 - 1.5 * iqr
    upper_bound = q3 + 1.5 * iqr

    outlier_mask = (numerical_frame[column] < lower_bound) | (numerical_frame[column] > upper_bound)

    numerical_frame_filtered = numerical_frame[~outlier_mask]

    return(numerical_frame_filtered)

# Part 0: Initialisation

In [ ]:
df = pd.read_csv("amz_uk_price_prediction_dataset.csv")

df.columns = [columns.lower().replace(" ", "_") for columns in df.columns]   ### Anti-whitespace pro snake_case
df.info()          

In [ ]:
display(df)

cat = df.select_dtypes(exclude="number")
print(f"The categorical columns of the {df} dataframe are :{list(cat.columns)}")

num = df.select_dtypes(include="number") 
print(f"The numerical columns of {df} dataframe are :{list(num.columns)}")

Numeric Data Check

In [ ]:
num_non_null_columns =num.notnull().sum()
num_non_null_total = num.notnull().sum().sum()
num_non_null_percentage_column = num.notnull().sum() / len(num)
num_non_null_percentage_total = (num.notnull().sum().sum()) / (num.size)
                                                                            # Count of non-missing values per column:

num_dup = num.duplicated()
num_dup_total = num.duplicated().sum()                                      # One fewer sum as duplicated returns series not a frame
num_na_columns = num.isna().sum()             
num_na_total = num.isna().sum().sum()

print(f"""
number of non-null values in the frame of numeric data: {num_non_null_total},
number of non-null values by column: \n\n{num_non_null_columns}\n

precentage of non-null values in the frame of numeric data: {num_non_null_percentage_total * 100},
precentage of non-null values by column: {num_non_null_percentage_column * 100}
      
number of duplicates in the frame of numeric data: {num_dup_total},

number of null values in the frame of numeric data:{num_na_total},
number of null values by column:\n\n{num_na_columns}\n""")

print(f"The following rows are duplicates: {','.join(num[num_dup].index )}") # num_dup is a boolean series which acts as a filter condition
print(f"The following columns have null values: {','.join(num_na_columns[num_na_columns > 0].index)}")


Catergoric Data Check

In [ ]:
##.strip() on cat

stripped = lambda _: _.strip() if isinstance(_, str) else _
cat = cat.map(stripped)

##

cat_non_null_columns = cat.notnull().sum()
cat_non_null_total = cat.notnull().sum().sum()
cat_non_null_percentage_column = cat.notnull().sum() / len(cat)
cat_non_null_percentage_total = (cat.notnull().sum().sum()) / (cat.size)
                                                                            # Count of non-missing values per column:

cat_dup = cat.duplicated()
cat_dup_total = cat.duplicated().sum()                                      # One fewer sum as duplicated returns series not a frame
cat_na_columns = cat.isna().sum()             
cat_na_total = cat.isna().sum().sum()

#cat_spaced_values_total_old = cat.eq(" ").sum().sum()                      # Old basic whitespace detector

cat_spaced_values_columns = cat.astype(str).apply(lambda _ : _.str.match(r'^\s*$')).sum() 
cat_spaced_values_total = cat.astype(str).apply(lambda _ : _.str.match(r'^\s*$')).sum().sum() 

# Treats every column as data type string, and for every column applies the check that it is not continuos whitespace using the regex
# expression r'^\s*$' and the str.match() method. ^ indicates the start, \s*checks for coninuos whitespace, and $indicates the end of the expression. 

print(f"""
number of non-null values in the frame of categoric data: {cat_non_null_total},
number of non-null values by column: \n\n{cat_non_null_columns}\n

percentage of non-null values in the frame of categoric data: {cat_non_null_percentage_total * 100},
percentage of non-null values by column: {cat_non_null_percentage_column * 100}

number of duplicates in the frame of categoric data: {cat_dup_total},

number of white space values in the frame of categoric data: {cat_spaced_values_total},
number of white space values by column: \n\n{cat_spaced_values_columns}

number of null values in the frame of categoric data:{cat_na_total},
number of null values by column:\n\n{cat_na_columns}\n""")

print(f"The following rows are duplicates: {','.join(cat[cat_dup].index )}") # cat_dup is a boolean series which acts as a filter condition
print(f"The following columns have null values: {','.join(cat_na_columns[cat_na_columns > 0].index)}")


# Part 1: Analyzing Best-Seller Trends Across Product Categories

## Crosstab Analysis

Are there categories where being a best-seller is more prevalent?

Creating a crosstable of "catergory" and "isbestseller" and the adding a 3rd column that is the proportion of bestsellers whithin the category

In [ ]:
cross_table_cat_bestsell = pd.crosstab(cat["category"], cat["isbestseller"])
cross_table_cat_bestsell["Proportion of best sellers in category"] = (cross_table_cat_bestsell.iloc[:,1]) / (cross_table_cat_bestsell.iloc[:,:2].sum(axis=1))


cross_table_cat_bestsell_sorted_by_true = cross_table_cat_bestsell.sort_values(by = True, ascending = False)
cross_table_cat_bestsell_sorted_by_proportion = cross_table_cat_bestsell.sort_values(by = "Proportion of best sellers in category", ascending = False)


print("Sorted by absolute number of bestsellers \n")
display(cross_table_cat_bestsell_sorted_by_true)

print("Sorted by proportion of bestsellers \n")
display(cross_table_cat_bestsell_sorted_by_proportion)


plotting top 10 of freq_table_cat_isbestseller

In [ ]:
plt.subplots(figsize = (10,5))
sns.barplot(x=cross_table_cat_bestsell_sorted_by_true.iloc[:25].index, y=cross_table_cat_bestsell_sorted_by_true.iloc[:25,1].values)
plt.xticks(rotation=90)
plt.xlabel('Product Categories')
plt.ylabel('Count')
plt.title("Absolute Count of Bestsellers by Category, Top 25")
plt.show()

plt.subplots(figsize = (10,5))
sns.barplot(x=cross_table_cat_bestsell_sorted_by_true.iloc[:10].index, y=cross_table_cat_bestsell_sorted_by_true.iloc[:10,1].values)
plt.xticks(rotation=90)
plt.xlabel('Product Categories')
plt.ylabel('Count')
plt.title("Absolute Count of Bestsellers by Category, Top 10")
plt.show()

plt.subplots(figsize = (10,5))
sns.barplot(x=cross_table_cat_bestsell_sorted_by_true.iloc[:2].index, y=cross_table_cat_bestsell_sorted_by_true.iloc[:2,1].values)
plt.xticks(rotation=90)
plt.xlabel('Product Categories')
plt.ylabel('Count')
plt.title("Absolute Count of Bestsellers by Category, Top 2")
plt.ylim(500,600)
plt.show()

So in terms of absolute number of bestsellers, "Grocery" and "Health & Personal Care" are dominant.

Now for the proportion of best sellers by category

In [ ]:
plt.subplots(figsize = (10,5))
sns.barplot(x=cross_table_cat_bestsell_sorted_by_proportion.iloc[:25].index, y=cross_table_cat_bestsell_sorted_by_proportion.iloc[:25,2].values)
plt.xticks(rotation=90)
plt.xlabel("Product Categories")
plt.ylabel("Proportion of best sellers in category")
plt.title("Proportion of Bestsellers Within Own Category, Top 25")
plt.show()

plt.subplots(figsize = (10,5))
sns.barplot(x=cross_table_cat_bestsell_sorted_by_proportion.iloc[:10].index, y=cross_table_cat_bestsell_sorted_by_proportion.iloc[:10,2].values)
plt.xticks(rotation=90)
plt.xlabel("Product Categories")
plt.ylabel("Proportion of best sellers in category")
plt.title("Proportion of Bestsellers Within Own Category, Top 10")
plt.show()

plt.subplots(figsize = (10,5))
sns.barplot(x=cross_table_cat_bestsell_sorted_by_proportion.iloc[:3].index, y=cross_table_cat_bestsell_sorted_by_proportion.iloc[:3,2].values)
plt.xticks(rotation=90)
plt.xlabel("Product Categories")
plt.ylabel("Proportion of best sellers in category")
plt.title("Proportion of Bestsellers Within Own Category, Top 3")
plt.ylim(0.05,0.06)
plt.show()

So in terms of the proportion of bestsellers withing their own category "Grocery" and "Health & Personal Care" are dominant again. With the addition of "Smart Home Security & Lighting".

Due to "Grocery" and "Health & Personal Care" being dominant in both senses would also suggest they are very large categories.

## Statistical Tests

Conduct a Chi-square test to determine if the best-seller distribution is independent of the product category.

In [ ]:
cross_table_cat_bestsell_raw = cross_table_cat_bestsell.drop(columns = "Proportion of best sellers in category")

chi2, p, dof, expected = chi2_contingency(cross_table_cat_bestsell_raw)

print("Chi2:", chi2)
print("p-value:", p)
print("Degrees of freedom:", dof)
print(expected)

a couple of points to take away form this 

- Our Chi square indepence statisitic is huge. this is due to the expected number of true results being incredibly low, being in single or double digits.
so to have some categories be in the hundreds, Deviates from the expeccted result in an extreme manner.
- p-value is not exactly 0, but it is extremely small (p < 10^(-300) , scipy's rounding point for 0). it is so low due to the huge deviations from execpted values. So whilst not exactly zero, the chance that this result or a result more extreme than this occuring by chance is *effectively* zero. So we can with great ease reject the hypothesis that bestseller status of a product and its category are independent of each other.

We have established that the relation is statistcally significant. Now we need to find out how strong that relation is.

This is where Cramér’s V comes into play giving us an insight into the stregnth of the relationship between best-seller status and product category.

In [ ]:
cram_v = association(cross_table_cat_bestsell_raw)
cram_v

This statisitic tells us that the relationship, while it is extremely statistically significant, is weak and may only have a minor impact on the dataset as whole.

## visualisations

Visualize the relationship between product categories and the best-seller status using a stacked bar chart.

In [ ]:
cross_table_cat_bestsell_sorted_by_true["Proportion of best sellers in category"]


stacked_proportion_df = pd.DataFrame({
    'True': cross_table_cat_bestsell_sorted_by_true["Proportion of best sellers in category"],
    'False': 1 - cross_table_cat_bestsell_sorted_by_true["Proportion of best sellers in category"]})

stacked_proportion_df

#= stacked_proportion_df.sort_values("True")



In [ ]:

ax = stacked_proportion_df[:50][['False', 'True']].sort_values("False").plot(
    kind='bar',                                                        
    stacked=True,
    figsize=(20, 10),
    color={
        'False': sns.color_palette("Set2")[0],
        'True': sns.color_palette("Set2")[1]
    },
    edgecolor='none'
)

ax.set_title("Proportion of Bestsellers for The 50 Most Populated Categroies")
ax.set_xlabel("Category")
ax.set_ylabel("Proportion of best sellers in category")
ax.set_ylim(0.9, 1)
ax.legend(title="Is Bestseller", labels=["No", "Yes"])
plt.xticks(rotation=90)
plt.tight_layout()
plt.show()


In [ ]:
ax = stacked_proportion_df[:][['False', 'True']].sort_values("False").plot(
    kind='bar',
    stacked=True,
    figsize=(80, 10),
    color={
        'False': sns.color_palette("Set2")[0],
        'True': sns.color_palette("Set2")[1]
    },
    edgecolor='none'
)

ax.set_title("Proportion of Bestsellers for all Categroies")
ax.set_xlabel("Category")
ax.set_ylabel("Proportion of Bestsellers")
ax.set_ylim(0.94, 1)
ax.legend(title="Is Bestseller", labels=["No", "Yes"])
plt.xticks(rotation=90)
plt.tight_layout()
plt.show()

It is very clear that some catergories do indeed have an effect on the proportion of bestseller productsin that category. But overall throughout the dataset the effect is not common and very weak considering the altered y axis.

# Part 2: Exploring Product Prices and Ratings Across Categories

Objective: Investigate how different product categories influence product prices.

## Preliminary Step: Remove outliers in product prices.

In [ ]:
price_category_df = pd.DataFrame({"price" : num["price"],
                                 "category" : cat["category"]})

In [ ]:
price_category_df_filtered = iqr_outlier_strip(price_category_df,"price")

price_category_df_filtered

## Violin Plots:

In [ ]:
top_20_categories_list = price_category_df_filtered["category"].value_counts().iloc[:20].index
top_10_categories_list = price_category_df_filtered["category"].value_counts().iloc[:10].index
top_5_categories_list = price_category_df_filtered["category"].value_counts().iloc[:5].index

top_20_categories_df = price_category_df_filtered[price_category_df_filtered["category"].isin(top_20_categories_list)]
top_10_categories_df = price_category_df_filtered[price_category_df_filtered["category"].isin(top_10_categories_list)]
top_5_categories_df = price_category_df_filtered[price_category_df_filtered["category"].isin(top_5_categories_list)]

Use a violin plot to visualize the distribution of price across different product categories. Filter out the top 20 categories based on count for better visualization.

Which product category tends to have the highest median price? Don't filter here by top categories.

In [ ]:
plt.figure(figsize = (10,10))
sns.violinplot(top_20_categories_df, y="category",x="price")
plt.xlabel('Prices')
plt.ylabel('Product Categories')
plt.title("Distribution of prices for each category in the Top 20 most populated")

plt.figure(figsize = (10,10))
sns.violinplot(top_10_categories_df, y="category",x="price")
plt.xlabel('Prices')
plt.ylabel('Product Categories')
plt.title("Distribution of prices for each category in the Top 10 most populated")

plt.figure(figsize = (10,10))
sns.violinplot(top_5_categories_df, y="category",x="price")
plt.xlabel('Prices')
plt.ylabel('Product Categories')
plt.title("Distribution of prices for each category in the Top 5 most populated")

plt.show()


In [ ]:
category_price_median = price_category_df_filtered.groupby("category")["price"].agg("median")
category_price_median = category_price_median.sort_values(ascending = False)

category_price_mean= price_category_df_filtered.groupby("category")["price"].agg("mean")
category_price_mean = category_price_mean.sort_values(ascending = False)

category_price_mean_top_10 = top_10_categories_df.groupby("category")["price"].agg("mean")
category_price_mean_top_10 = category_price_mean_top_10.sort_values(ascending = False)


In [ ]:
plt.figure(figsize = (10,5))

sns.barplot( x = category_price_median[:20].index, y = category_price_median[:20].values)
plt.xticks(rotation=90)
plt.xlabel('Product Categories')
plt.ylabel('Median Price')
plt.title("Median Price of product categories, 20 highest")
plt.show()

## Bar Charts:

In [ ]:
plt.figure(figsize = (10,5))

sns.barplot(x = category_price_mean_top_10.index, y = category_price_mean_top_10.values)
plt.xticks(rotation=90)
plt.xlabel('Product Categories')
plt.ylabel('Mean Price')
plt.title("Mean Price of top 10 most populated categories")
plt.show()

In [ ]:
plt.figure(figsize = (10,5))

sns.barplot(x = category_price_mean[:10].index, y = category_price_mean[:10].values)
plt.xticks(rotation=90)
plt.xlabel('Product Categories')
plt.ylabel('Mean Price')
plt.title("Mean Price of products by category, 10 Highest")
plt.show()

## Box Plots:

In [ ]:
star_category_df = pd.DataFrame({"stars": num[num["stars"] >= 1]["stars"],
                          "category": cat["category"]})

star_category_top_10_df = star_category_df[star_category_df["category"].isin(top_10_categories_list)]

In [ ]:
order_by_median = (
    star_category_top_10_df.groupby("category")["stars"]
    .agg("median")
    .sort_values(ascending = False)
    .index)


plt.figure(figsize = (10,10))

sns.boxplot(
    star_category_top_10_df ,
    x = star_category_top_10_df["stars"],
    y = star_category_top_10_df["category"],
    order = order_by_median,
    showfliers = False) 

plt.xlabel('Star Rating')
plt.ylabel('Categories')
plt.title("Box Plot of top 10 Largest Categories")

Out of the ten largest categories, "Bath & Body" has the highest median star rating.

In [ ]:
order_by_median = (
    star_category_df.groupby("category")["stars"]
    .agg("median")
    .sort_values(ascending = False)
    .index)


plt.figure(figsize = (10,80))
sns.boxplot(
    star_category_df ,
    x = star_category_df["stars"],
    y = star_category_df["category"],
    order = order_by_median,
    showfliers = False) 

plt.xlabel('Star Rating')
plt.ylabel('Categories')
plt.title("Box Plot of All Categories")

Plotting again, but only with the highest ten medians for clarity:


In [ ]:
order_by_median = (
    star_category_df.groupby("category")["stars"]
    .agg("median")
    .sort_values(ascending = False)
    .index)

top_10_catergories = order_by_median[:10]

plt.figure(figsize = (10,10))
sns.boxplot(
    (star_category_df[:10]) ,
    x = star_category_df["stars"],
    y = star_category_df["category"],
    order = top_10_catergories,
    showfliers = False) 
plt.xlabel('Star Rating')
plt.ylabel('Categories')
plt.title("Box Plot of the top 10 categories in terms of Median")

Out of the top 10 most populated categories, "Handmade Clothing, Shoes & Accessories" has the highest median.

In [ ]:
star_category_df.groupby("category")["stars"].agg("median").sort_values(ascending = False)


However for considering all categories there is a four-way tie for highest median between:

- Handmade Artwork                          5.0 Stars
- Handmade Baby Products                    5.0 Stars
- Made in Italy Handmade                    5.0 Stars
- Snowboards                                5.0 Stars

With Handmade Clothing, Shoes & Accessories  coming in 5th

- Handmade Clothing, Shoes & Accessories    4.9 Stars

# Part 3: Investigating the Interplay Between Product Prices and Ratings

Objective: Analyze how product ratings (stars) correlate with product prices.

## Correlation Coefficients:

In [ ]:
price_stars_df = num[num["stars"] >= 1][["stars","price"]]
price_stars_df 

#### remove outliers #### 

price_stars_df_filterd = iqr_outlier_strip(price_stars_df,"price")
price_stars_df_filterd

In [ ]:
pearson_correlation = price_stars_df_filterd.corr(method = "pearson")
spearman_corrrelation = price_stars_df_filterd.corr(method = "spearman")
kendall_correlation = price_stars_df_filterd.corr(method = "kendall")

print(f"""
pearson_correlation = \n{price_stars_df_filterd.corr(method = "pearson")}\n
spearman_corrrelation = \n{price_stars_df_filterd.corr(method = "spearman")}\n
kendall_correlation = \n{price_stars_df_filterd.corr(method = "kendall")}\n""")

There appears to be no meaningful correlation of any kind  between star rating and price

## Visualizations:

In [ ]:
sns.scatterplot(x =price_stars_df["stars"], y = price_stars_df["price"])

plt.xlabel('Star')
plt.ylabel('price')
plt.xlim(1,5)

plt.title("Star rating and price scatterplot, Outliers Remaining")

plt.show()

Excluding the minimum and maximum ratings, which seem to be behavioural defaults, there does seem to be a mild trend that if a customer purchases a more expenisve product they are more likely to review the product more favourably. bring the extreme values back into the picture, it may simply be the case that more expnsive products result in more extrme reveiws due to expectations.

In [ ]:
sns.scatterplot(x =price_stars_df_filterd["stars"], y = price_stars_df_filterd["price"])

plt.xlabel('Star')
plt.ylabel('price')
plt.xlim(1,5)
plt.title("Star rating and price scatterplot, Outliers Removed")

plt.show()

When you remove the outliers, there appears to be zero correlation between star price and review. It would appear that sometimes outliers offer insights that refined data doesn't. Or just that truncated data inspires speculation.

In [ ]:
# #data binning may be a good idea for a crosstable that becomes a heat map

# price_bins = [0,10,25,50,100,250,500,1000,10000]
# labels_bins = ["0-10","10-25","25-50","50-100","100-250","250-500","500-1000","1000-10000"]

# price_stars_df["grouped_prices"] = pd.cut(price_stars_df["price"],
#                                           bins = price_bins,
#                                           labels = labels_bins,
#                                           include_lowest=True)

# price_stars_df 


ALL NUMERICAL VARIABLES

In [ ]:
num_all_outliers_removed = num.copy()

for _ in num_all_outliers_removed:
    num_all_outliers_removed = iqr_outlier_strip(num_all_outliers_removed,str(_))

#yes, map() is also possible

In [ ]:
print(num.corr())


sns.heatmap(num.corr())
plt.title("Correlation Heatmap of Numeric Values, Outliers included")
plt.show()

print(num_all_outliers_removed.corr())

sns.heatmap(num.corr())
plt.title("Correlation Heatmap of Numeric Values, Outliers for all series Removed")
plt.show()

No signifcant correlation exists in any combination of numeric variables before and after outlier removal. If you need to find a trace of correlation maybe there exists somethin between bought in the last month and stars and the number of reviews. But the correlation is still too weak to draw anything meaningful from this.

In [ ]:
probplot(num_price_filtered["price"],dist="norm",plot = plt)
plt.title("QQ plot to show if product price is normally distributed, outliers removed")



The quantile quantile plot has a couple of features to break down:
- There is a large S shape, which normally suggests heavy tails in the distribution. But, the flat  horizontal begining and end of our qq plot suggests that there is truncation of the data which actually would result in lighter shorter tails. 
-The observation abouttruncation appears to be valid as prices can not be negative and our outlier filtering has removed the possibility of values above an upper bound which appears around 100
- Although self-evident it is still worth pointing out that our price distribution deviates significantly from a normal distribution.




In [ ]:
probplot(num["price"],dist="norm",plot = plt)
plt.title("QQ plot to show if product price is normally distributed, Outliers Included")
plt.ylim(1000)
plt.show()

probplot(num["price"],dist="norm",plot = plt)
plt.title("QQ plot to show if product price is normally distributed, Outliers Included")
plt.ylim(0,1000)
plt.show()

the left side of the plot remains the same as before however our price distribution deviates from the values of the theoretical normal distribution almost vertically. This right side our prices distribution is no longer showing signs of truncation near 100 and continues onwards creating a very heavy tail relative to the tail of the theoretical normal distribution.